# TinyChirp Torch + LiteRT build pipeline

Trains one Torch CNN on log-mel spectrograms, exports an int8 TFLite model (PT2E per-tensor quantization with representative calibration, analogous to the TensorFlow CNN mel notebook), and writes a Rust audio sample file.

In [1]:
from pathlib import Path
import random
import numpy as np
import librosa
import torch
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR = REPO_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = REPO_ROOT / "dataset"

OUT_TFLITE = MODELS_DIR / "cnn_mel_torch.tflite"
OUT_AUDIO_RS = SRC_DIR / "audio_sample.rs"

SAMPLE_RATE = 16000
FRAME_LENGTH = 1024
FRAME_STEP = 256
NUM_MEL_BINS = 80
LOWER_EDGE_HERTZ = 80.0
UPPER_EDGE_HERTZ = 8000.0
TARGET_FRAMES = 184
TARGET_AUDIO_LEN = (TARGET_FRAMES - 1) * FRAME_STEP + FRAME_LENGTH

# Keep training on CPU to avoid JAX-backed device runtime conflicts.
DEVICE = "cpu"
print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)
print("Device:", DEVICE)


Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/cnn_mel_torch.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs
Device: cpu


In [2]:
def load_and_fix_audio(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if audio.shape[0] >= TARGET_AUDIO_LEN:
        audio = audio[:TARGET_AUDIO_LEN]
    else:
        pad = TARGET_AUDIO_LEN - audio.shape[0]
        audio = np.pad(audio, (0, pad), mode="constant")
    return audio.astype(np.float32)


def create_log_mel_spectrogram(audio: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=FRAME_LENGTH,
        hop_length=FRAME_STEP,
        win_length=FRAME_LENGTH,
        n_mels=NUM_MEL_BINS,
        fmin=LOWER_EDGE_HERTZ,
        fmax=UPPER_EDGE_HERTZ,
        center=False,
        window="hann",
        power=2.0,
    )
    mel = np.log(mel + 1e-6)
    if mel.shape[1] > TARGET_FRAMES:
        mel = mel[:, :TARGET_FRAMES]
    elif mel.shape[1] < TARGET_FRAMES:
        mel = np.pad(mel, ((0, 0), (0, TARGET_FRAMES - mel.shape[1])), mode="constant")
    return mel.T.astype(np.float32)


def scan_split(split: str):
    base = DATASET_ROOT / split
    class_dirs = sorted([p for p in base.iterdir() if p.is_dir()])
    class_names = [p.name for p in class_dirs]
    label_to_idx = {name: i for i, name in enumerate(class_names)}
    samples = []
    for class_dir in class_dirs:
        for wav in sorted(class_dir.glob("*.wav")):
            samples.append((wav, label_to_idx[class_dir.name]))
    return samples, class_names


In [3]:
class TinyChirpDataset(Dataset):
    def __init__(self, split: str, class_names=None):
        samples, found_class_names = scan_split(split)
        self.samples = samples
        self.class_names = class_names or found_class_names

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav_path, label = self.samples[idx]
        audio = load_and_fix_audio(wav_path)
        spec = create_log_mel_spectrogram(audio)
        spec = np.expand_dims(spec, axis=0)  # [1, frames, mel]
        return torch.from_numpy(spec), torch.tensor(label, dtype=torch.long), str(wav_path)


train_samples, class_names = scan_split("training")
num_labels = len(class_names)
print("Classes:", class_names)
print("Training clips:", len(train_samples))

train_ds = TinyChirpDataset("training", class_names=class_names)
val_ds = TinyChirpDataset("validation", class_names=class_names)
test_ds = TinyChirpDataset("testing", class_names=class_names)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


Classes: ['non_target', 'target']
Training clips: 11292


In [4]:
class TinyChirpCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2),
            nn.Conv2d(4, 4, kernel_size=3),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4 * 44 * 18, 8),
            nn.ReLU(),
            nn.Linear(8, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = TinyChirpCNN(num_classes=num_labels).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb, _ in tqdm(loader):
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += float(loss.item()) * yb.size(0)
            pred = logits.argmax(dim=1)
            correct += int((pred == yb).sum().item())
            total += yb.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


EPOCHS = 1
for epoch in range(EPOCHS):
    model.train()
    for xb, yb, _ in tqdm(train_loader):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    train_loss, train_acc = evaluate(train_loader)
    val_loss, val_acc = evaluate(val_loader)
    print(
        f"epoch={epoch + 1} train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

test_loss, test_acc = evaluate(test_loader)
print(f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}")


100%|██████████| 44/44 [00:28<00:00,  1.55it/s]


epoch=1 train_loss=0.1645 train_acc=0.9286 val_loss=0.1496 val_acc=0.9312


100%|██████████| 44/44 [00:25<00:00,  1.75it/s]

test_loss=0.1424 test_acc=0.9354


In [5]:
import litert_torch
from litert_torch.quantize import quant_config as qcfg
from litert_torch.quantize import pt2e_quantizer
from torchao.quantization.pt2e import quantize_pt2e

model.eval()
sample_inputs = (torch.randn(1, 1, TARGET_FRAMES, NUM_MEL_BINS, device=DEVICE),)

# Full int8 post-training quantization with per-tensor scales (symmetric config
# with is_per_channel=False), analogous to TFLite full integer with per-tensor
# scales in building_tensorflow/cnn_mel.ipynb (representative_dataset + int8 I/O).
exported = torch.export.export(model.cpu(), (sample_inputs[0].cpu(),))
m = exported.module()

quantizer = pt2e_quantizer.PT2EQuantizer().set_global(
    pt2e_quantizer.get_symmetric_quantization_config(is_per_channel=False)
)
m = quantize_pt2e.prepare_pt2e(m, quantizer)

# Representative calibration (same idea as TF: take=100 from the test loader).
with torch.no_grad():
    for i, (xb, _, _) in enumerate(test_loader):
        if i >= 100:
            break
        m(xb.cpu())

m = quantize_pt2e.convert_pt2e(m, fold_quantize=False)
m.eval()

quant_cfg = qcfg.QuantConfig(pt2e_quantizer=quantizer)
edge_model = litert_torch.convert(m, (sample_inputs[0].cpu(),), quant_config=quant_cfg)

with torch.no_grad():
    torch_q_out = m(sample_inputs[0].cpu()).detach().cpu().numpy()
edge_output = edge_model(sample_inputs[0].cpu())

if np.allclose(torch_q_out, edge_output, atol=1e-2, rtol=1e-2):
    print("Quantized Torch vs LiteRT was within tolerance")
else:
    print("Quantized Torch vs LiteRT differed beyond tolerance")

edge_model.export(str(OUT_TFLITE))
print("Wrote int8 per-tensor TFLite to", OUT_TFLITE)


I0000 00:00:1775229037.171455   66994 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775229037.363399   66994 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775229039.214703   66994 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.0             Please see http

AssertionError: Guard failed: x.size()[0] == 1